# Phân loại SMS: Naive Bayes, Logistic Regression và Dense Network
Notebook chạy source trong gói đính kèm. Chọn Runtime → Run all cho phần cơ bản; neural bật riêng ở cell cuối. Spam là positive class; cùng split seed=42, test=20%.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Chọn sms_spam_lab.zip
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    root = Path('/content/sms_spam_lab').resolve()
    for item in z.infolist():
        if not (root / item.filename).resolve().is_relative_to(root):
            raise ValueError('Unsafe ZIP path')
    z.extractall(root)
%cd /content/sms_spam_lab
%pip install -q -r requirements.txt

## 1–2. Pandas và chuẩn hóa NLTK
Lemmatization có POS rồi stemming; không fit bất kỳ thống kê nào trên test. Đọc thêm giải thích chi tiết trong README.md.

In [ ]:
from spam_lab.preprocessing import setup_resources, Normalizer
from spam_lab.train import load_data
setup_resources()
df, total = load_data('data/smsspamcollection.tsv')
display(df.head())
display(df.label.value_counts())
print(Normalizer()('Cats were running. Call 123 now!'))

## 3–8. Train/test, vocabulary train, NB bằng Counter và hai baseline LR
Dùng thư mục có thời gian để có thể chạy lại cell mà không ghi đè kết quả trước.

In [ ]:
from datetime import datetime
from spam_lab.train import train
output = 'colab_artifacts_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
report = train('data/smsspamcollection.tsv', output=output)
report['metadata']

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(f'{output}/comparison.csv'))
display(Image(filename=f'{output}/confusion_matrices.png'))
display(pd.read_csv(f'{output}/nb_vocabulary.csv').head(20))

Confusion matrix: hàng là nhãn thật, cột là dự đoán, thứ tự ham/spam. Accuracy=(TP+TN)/N; precision=TP/(TP+FP); recall=TP/(TP+FN). So sánh F1 và recall bên cạnh accuracy vì mất cân bằng lớp.

In [ ]:
from spam_lab.inference import Predictor
predictor = Predictor(output)
predictor.predict(['You won free cash! Call now', 'See you tonight'], explain=True)

## 9*. Dense Network tùy chọn
Đổi RUN_NEURAL=True. Validation tách từ train, không dùng test để EarlyStopping.

In [ ]:
RUN_NEURAL = False
if RUN_NEURAL:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-neural.txt'])
    neural_output = output + '_neural'
    neural_report = train('data/smsspamcollection.tsv', output=neural_output, neural=True)
    display(pd.read_csv(f'{neural_output}/comparison.csv'))

In [ ]:
import shutil
shutil.make_archive(output, 'zip', output)
files.download(output + '.zip')

## Nộp bài và demo
Lưu notebook vào Drive rồi Share để lấy link Colab. Dùng kết quả chạy thực tế và confusion matrix để viết báo cáo PDF. Backend local: `python run_demo.py`; hướng dẫn API và frontend fetch có trong README.md.